# Singularity

容器：一个允许我们在资源隔离的过程中，运行应用程序和其依赖项的、轻量的、操作系统级别的虚拟化技术

容器 vs．虚拟机：
- 虚拟机：与主机服务器共享硬件资源；独立操作系统；占用全部预分配资源；
- 容器：与主机服务器共享硬件资源和操作系统；资源动态分配；

Singularity vs．Docker：
- Docker：使用广泛，庞大用户社区（Docker Hub）；依赖root权限
- Singularity：兼容性好，用户群体增长迅速；无需root权限

## 预编译镜像

从Docker Hub(docker://)、Singularity Hub(shub://拉取镜像)

```shell
singularity pull IMAGE.sif docker://IMAGE

```

```shell

# 从singularity库下载容器
$ singularity pull library://godlovedc/funny/lolcow
# 下载完成后，将会看见一个sif结尾的文件，这个就是singularity的容器文件
lolcow_latest.sif
# 也可以从docker库下载容器
$ singularity pull docker://godlovedc/lolcow
```

＂交我算＂超算Docker Hub主页【https://hub.docker.com/u/sjtuhpc】（docker://sjtuhpc/）

```shell
unset XDG_RUNTIME_DIR
unset SINGULARITY_BIND
unset MODULEPATH
singularity pull lammps-intel-2020.sif docker://sjtuhpc/hpc-app-container:lammps-intel-2020
```




## Singularity容器使用
```shell
singularity shell lolcow_latest.sif # 进入容器：shell命令 退出容器：exit命令

## 运行容器：run命令
## 此处的run，实际上是调用容器内的一个runscript的特殊文件
singularity run IMAGE.sif
 ## 在容器内执行任务：exec命令
singularity exec IMAGE.sif COMMAND

# 检查容器内的runscript文件
## singularity inspect --runscript lolcow_latest.sif


singularity instance start：启动一个容器实例。
singularity build：构建一个 SIF 容器映像。

```

<!-- #!/bin/sh

    fortune | cowsay | lolcat -->

此外，也可以像运行软件一样来使用容器，例如

`./lolcow_latest.sif`

## 创建、删除容器实例

我们可以在.sif镜像文件基础上，创建一个容器（singularity instance start），运行容器（singularity run），进入容器（singularity shell)，停止容器（singularity instance stop）。

```shell
singularity instance start /tmp/my-sql.sif mysql
 
 
singularity shell instance://mysql
# Singularity my-sql.sif> pwd
 
 
singularity instance stop /tmp/my-sql.sif mysql
```


## 要构建一个容器，必须使用build命令

一个标准的容器开发周期为：

1. 编写一个可写的容器（sandbox）      
2. 使用--writeable进入容器，以交互方式对容器进行修改    
3. 对文件进行修改       
4. 重新build    
5. 重复前两步，直到容器变成我们期望的状态   
6. build并保存容器，以供生产使用    

例如我们手头已经有一个现成的recipes（用于build的命令文件）


定义文件分为两部分：

- Header：描述了要在容器内构建的核心操作系统。例如，它可以指定要从哪个基础映像开始构建。
    - Bootstrap：这个关键字定义了基础镜像的来源、协议或格式。你可以选择多个来源，包括 Singularity 容器库（library）、Docker Hub、OCI Registry as Storage（ORAS）、yum、本地镜像（localimage）、debootstrap、开放容器倡议（OCI）、BusyBox 等等。
    - From：这个关键字依赖于 Bootstrap 的值。



```shell
BootStrap: debootstrap
OSVersion: stable
MirrorURL: http://ftp.us.debian.org/debian/

%runscript
    echo "This is what happens when you run the container..."

%post
    echo "Hello from inside the container"
    apt-get -y --allow-unauthenticated install vim
```

在这个文件中，%开头的通常表示在构建过程中起着一定特殊作用，具体可以参考[文档](https://docs.sylabs.io/guides/3.5/user-guide/definition_files.html)


- Sections：这些是一组命令，用于描述在最终映像中的特定动作。包括设置环境、复制文件、设置环境变量、下载文件、进行测试等等。
    - %files：这个部分允许你复制文件进入容器。这在程序编译完成后，你想将其移动到第二阶段容器中时非常有用。
    - %environment：这个部分允许你定义运行时会设置的环境变量。这些变量在构建时不可用。
    - %post：这里可以使用像 git 和 wget 这样的工具从互联网下载文件，安装新软件，编写配置文件，在容器内创建目录。
    - %runscript：这个部分包含了写入到容器内部文件的命令，这些命令会在使用 singularity run 命令运行容器映像时执行。
    - %startscript：这个部分类似于 %runscript，不同之处在于，%startscript 中的命令会在执行 singularity instance start 命令时运行。
    - %test：这个部分包含了在构建过程结束时执行的命令。它通常包含你选择的验证命令，也会在使用 singularity test 命令时执行。
    - %labels：这个部分用于向文件 /.singularity.d/labels.json 添加元数据，标签通过键值对定义，并可以在执行 singularity inspect 命令时显示。
    - %help：这部分的文本会被转移到容器内的一个元数据文件中。可以使用 singularity run -help 命令显示这些帮助信息。

常用的符号有：

- %post(执行命令，如下载、安装、编写配置文件、创建目录等)
- %files(将文件复制到容器中)
- %test(构建完成后，测试容器)
- %environment(设置环境变量，类似于修改.bashrc)
- %runscript(比较特殊，在呼出容器时，可以自动执行计划的任务)




## 从头构建一个容器
对于新手而言，通常在build容器时，需要给出--sandbox选项，这在我们还不知道容器开发具体需要包含什么文件的时候，非常有用

开始build容器

```shell
 sudo singularity build --sandbox lolcow lolcow.def
```

运行完成后，会在当前目录下看到一个lolcow文件

对容器进行探索和修改

```shell
sudo singularity shell --writable lolcow
Singularity> apt-get update
Singularity> apt-get install -y fortune cowsay lolcat
```

此时可以看到应用程序成功安装，但如果没有使用--writeable参数，那么在进入容器后，执行安装命令，通常会被告知这些文件都是read-only，不可修改

需要说明的是，与docker类似，在容器中进行的操作，在退出容器后，这些修改并不会保存，如果要使这些修改永久生效，就需要将这些操作加入到定义文件中，重新build







### 从现有容器构建
从现有容器构建，需要注意的是，要修改build定义文件的header

例如需要以容器库中现有的容器作为起点

```shell
# 例如以debian为起点
BootStrap: library
From: debian
# 或者从docker hub上的debian启动
Bootstrap: docker
From: debian
# 再或者以本地文件系统上的基础容器启动
Bootstrap: localimage
From: /home/test/debian.sif

```

与从头构建容器相比，从现有容器构建的最大好处是不需要root权限

`singularity build debian3.sif debian2.sif`


### 安全的构建容器
在singularity中，提供了--fakeroot选项，可以以更安全的方式在容器中使用root权限来build容器

```shell
singularity build --fakeroot container.sif container.def
```

fakeroot 允许用户在容器及其请求的命名空间内拥有与 root 用户几乎相同的管理权限。这意味着用户可以执行通常需要 root 权限的操作，而不需要实际的 root 权限。通过 fakeroot 用户创建的所有文件或目录，在容器内部归 root 所有，在容器外部则归创建它们的用户和组所有。




### 容器挂载主机文件
默认情况下，容器运行后会自动绑定主机的几个目录，包括：

```shell
$HOME

/tmp

/proc

/sys

/dev

```
此外，也可以使用`--bind/-B/SINGULARITY_BINDPATH`来指定需要绑定的其他目录

至此，你已经掌握了构建一个singularity容器的必须技能了，可以去尝试构建一个属于自己的容器。

<img src="https://pica.zhimg.com/v2-5a3dd59c572c85d4c1544734ec124532_1440w.jpg">


## 交互式shell构建镜像
### 容器构建节点：
```shell
ssh build＠container-x86
ssh build＠container-arm
```
### 创建工作目录：
```shell
cd \＄（mktemp－d）
```
### 下载基础操作系统镜像：

```shell
docker pull centos:8
docker images
docker run -name=MY_NAME IMAGE_ID /bin/bash # root身份进入容器内
```

### 以root特权修改容器内容：
```shell
yum install SOFTWARE
...
exit
```
### 提交容器变更：
```shell
docker commit CONTAINER_ID IMG_NAME
```
### 保存singularity镜像：
```shell
SINGULARITY_NOHTTPS＝1 singularity build IMG_NAME.sif docker-daemon://IMG_NAME:latest
```



## 通过Definition File构建镜像
- 进入容器构建节点\＆创建工作目录与交互式shell构建镜像一致
- 镜像自定义文件 sample．def
```shell
Bootstrap: docker
From: centos:8
%help
    This is a demo
%environment
    export MYPATH=xxx/yyy
%post
    yum install SOFTWARE
%runscript
    echo "SOFTWARE is installed"
```
- 构建镜像（在container－x86或者container－arm上）：

这条命令使用 Docker 运行一个 Singularity 容器，并在容器内执行 singularity build 来构建一个 Singularity 镜像（.sif 文件）。下面是对命令的详细解析：

```shell
docker run --privileged --rm -v \
    ${PWD}:/home/singularity \
    sjtuhpc/centos7-singularity:x86 \
    singularity build /home/singularity/sample-x86.sif /home/singularity/sample.def
```